# 15 · Membandingkan XGBoost dan Temporal GNN
**Pertanyaan penelitian.** Pada dua detik pertama setelah kehilangan bola, seberapa baik kedua model mendeteksi gegenpressing pada pertandingan yang disisihkan dari pelatihan?

Pilih kernel Python dari folder `venv` proyek, kemudian jalankan sel dari atas ke bawah. Percobaan awal memakai label yang dibekukan pada 27 Oktober 2026. Untuk eksperimen artikel, arahkan `ANNOTATIONS` ke hasil audit poin 2. Notebook hanya membaca data sumber dan membuat folder keluaran baru untuk setiap pelatihan.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "match_comparison.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.match_comparison import MATCHES, SEED, load_inputs, run_comparison

PROCESSED = ROOT / "data" / "processed"
ANNOTATIONS = ROOT / "artifacts" / "final_training_20261027T152829Z" / "frozen_annotations.csv"
OUTPUT = ROOT / "artifacts" / "perbandingan_dua_model"
MODE = "uji_coba"  # pilihan lain: "periksa" atau "tujuh_pertandingan"
AUDIT_LABEL_SELESAI = False  # ubah setelah memeriksa sumber label untuk eksperimen utama

## 1. Periksa episode dan pembagian pertandingan
Setiap pertandingan menjadi data uji sekali. Pertandingan berikutnya dalam tabel menjadi data validasi, dan lima lainnya menjadi data latih. Label ragu dikeluarkan. Pastikan jumlah kelas sesuai dengan versi anotasi yang dipilih.

In [2]:
inputs = load_inputs(PROCESSED, ANNOTATIONS)
_, episodes, _, _ = inputs
counts = pd.crosstab(episodes.match_id, episodes.gegenpressing_label).reindex(MATCHES)
counts.columns = ["negatif", "positif"]
counts["total"] = counts.sum(axis=1)
counts["validasi"] = list(MATCHES[1:]) + [MATCHES[0]]
print(f"{len(episodes)} episode, seed {SEED}. ID pada indeks adalah pertandingan uji.")
display(counts)

1873 episode, seed 42. ID pada indeks adalah pertandingan uji.


,negatif,positif,total,validasi
match_id,,,,
J03WMX,67,341,408,J03WN1
J03WN1,6,27,33,J03WOH
J03WOH,36,262,298,J03WOY
J03WOY,46,296,342,J03WPY
J03WPY,67,256,323,J03WQQ
J03WQQ,32,163,195,J03WR9
J03WR9,38,236,274,J03WMX


## 2. Jalankan kedua model
* **periksa** menampilkan data dan jadwal tanpa pelatihan.
* **uji_coba** melatih XGBoost dan Temporal GNN pada sampel kecil dari tujuh laga. Hanya satu putaran dan satu epoch GNN. Gunakan untuk memeriksa alur dan keluaran.
* **tujuh_pertandingan** melatih ulang kedua model dari awal pada setiap putaran. Jalankan setelah audit label. GNN memakai maksimal 100 epoch dengan penghentian dini berdasarkan loss validasi (patience 15). Setiap putaran menyimpan model dan prediksi di folder pertandingan.

XGBoost memakai 12 fitur tekanan dari notebook 09. GNN memakai posisi, kecepatan dan identitas simpul pemain/bola untuk 51 frame (0 hingga +2 detik, 25 Hz). Graf dipotong sebelum normalisasi. Kedua model memakai kelas target yang sama. Ambang dicari pada data validasi dari 0,01 hingga 0,99 untuk memaksimalkan macro-F1.

In [3]:
if MODE not in {"periksa", "uji_coba", "tujuh_pertandingan"}:
    raise ValueError("Periksa ejaan MODE.")
if MODE == "tujuh_pertandingan" and not AUDIT_LABEL_SELESAI:
    raise ValueError("Selesaikan audit poin 2, pilih berkas ANNOTATIONS yang telah diperiksa, lalu set AUDIT_LABEL_SELESAI = True.")
if MODE != "periksa":
    run_dir, comparison = run_comparison(inputs, PROCESSED, OUTPUT, MODE)
    print("Hasil", MODE, "tersimpan di", run_dir)
    display(comparison.round(4))

Folder hasil: c:\VAEP-Track\artifacts\perbandingan_dua_model\uji_coba_20261102T154447163291Z
Putaran 1: uji J03WMX, validasi J03WN1
Hasil uji_coba tersimpan di c:\VAEP-Track\artifacts\perbandingan_dua_model\uji_coba_20261102T154447163291Z


model,temporal_gnn,xgboost,selisih_gnn_minus_xgboost
match_id,,,
J03WMX,0.4958,0.4375,0.0583
rata_rata_bobot_setara,0.4958,0.4375,0.0583


## 3. Baca hasil per pertandingan
Selisih positif pada tabel sebelumnya berarti macro-F1 GNN lebih tinggi dalam laga tersebut. Baris terakhir memberi bobot sama untuk tiap laga yang diuji. Untuk mode penuh, periksa pula recall kelas negatif dan positif beserta jumlah kesalahannya. J03WN1 memiliki sedikit episode negatif. Gunakan metrik dari folder berstatus selesai dengan mode tujuh pertandingan untuk analisis artikel.

In [4]:
if MODE != "periksa":
    metrics = pd.read_csv(run_dir / "metrik_per_pertandingan.csv")
    display(metrics.round(4))
    if MODE == "uji_coba":
        print("Uji teknis selesai. Angka di atas berasal dari subset kecil.")

,match_id,model,n,tn,fp,fn,tp,recall_0,recall_1,macro_f1,roc_auc
0,J03WMX,temporal_gnn,12,6,0,5,1,1.0000,0.1667,0.4958,0.8333
1,J03WMX,xgboost,12,5,1,5,1,0.8333,0.1667,0.4375,0.6111


Uji teknis selesai. Angka di atas berasal dari subset kecil.
